# B1-R Retraining Campaign v1.0 — MASTER

Frozen corrected-benchmark retraining campaign for the four-model family.

**Scientific contract:** B1-R SHA `9fab73d477624ff3878ee3ec3cb2e86f5a4aa9bb17ce00e47fedf260d350fce7`; Training Protocol SHA `1ea5a907f4e0f7d5bc87d43bd82f649089d5350329afb0133d3de86f1ca33dab`. The notebook runs a seed-2026 Raw TS-ResNet/Raw SCAL-DAS smoke/verification gate first, then the resume-safe 4-model × 3-seed full campaign. Test data are not evaluated during smoke or before the 60-epoch training trajectory is complete and the primary validation-NLL checkpoint is frozen.

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess, shutil, base64, zipfile

if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Drive already available.')

subprocess.run([sys.executable,'-m','pip','install','-q','pandas==2.2.3','h5py>=3.11,<4','scikit-learn>=1.5,<2'],check=True)
import pandas as pd, numpy as np, torch, platform, sklearn, h5py

PROJECT_ROOT=Path('/content/drive/MyDrive/Project_Paper01_SCAL_DAS')
OUTPUT_ROOT=PROJECT_ROOT/'B1R_Retraining_Campaign_v1_0'
LOCAL_PACKAGE=Path('/content/b1r_campaign_v1_0')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
if not PROJECT_ROOT.is_dir(): raise FileNotFoundError(PROJECT_ROOT)
if not torch.cuda.is_available(): raise RuntimeError('GPU runtime is required. In Colab: Runtime > Change runtime type > GPU.')
print('GPU:',torch.cuda.get_device_name(0))
print('Project:',PROJECT_ROOT)
print('Output :',OUTPUT_ROOT)
env={'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'cuda_runtime':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),'pandas':pd.__version__,'numpy':np.__version__,'sklearn':sklearn.__version__,'h5py':h5py.__version__}
(OUTPUT_ROOT/'RUN_ENVIRONMENT.json').write_text(json.dumps(env,indent=2,sort_keys=True))
print(json.dumps(env,indent=2))


In [ ]:
# Reconstruct the frozen campaign implementation embedded in this notebook.
PAYLOAD_B64 = (
'...'
'BAAAAA=='
)
if LOCAL_PACKAGE.exists(): shutil.rmtree(LOCAL_PACKAGE)
LOCAL_PACKAGE.mkdir(parents=True,exist_ok=True)
payload_path=LOCAL_PACKAGE/'payload.zip'
payload_path.write_bytes(base64.b64decode(PAYLOAD_B64))
with zipfile.ZipFile(payload_path) as z: z.extractall(LOCAL_PACKAGE)
if str(LOCAL_PACKAGE) not in sys.path: sys.path.insert(0,str(LOCAL_PACKAGE))
from b1r_campaign import CampaignConfig, bootstrap_campaign, run_smoke, run_full, aggregate_campaign, make_audit_zip
from b1r_campaign.data import auto_detect_phase2
from b1r_campaign.train import initialization_equivalence_gate
from b1r_campaign.models import RDFTPreprocessor, build_model
cfg=CampaignConfig()
print('Embedded package ready:',cfg.version)


In [ ]:
# Preflight: locate Phase-2, decide whether local staging is safe, then verify B1-R.
phase2_candidate=auto_detect_phase2(PROJECT_ROOT)
required_rels=sorted(set(pd.read_csv(LOCAL_PACKAGE/'frozen'/'B1R_SAMPLE_ASSIGNMENTS.csv').shard_file.astype(str)))
required_bytes=sum((phase2_candidate/r).stat().st_size for r in required_rels)
free_bytes=shutil.disk_usage('/content').free
STAGE_LOCAL = free_bytes > required_bytes + 10*(1024**3)
print(f'Required shard bytes: {required_bytes/1024**3:.2f} GiB')
print(f'Local free space    : {free_bytes/1024**3:.2f} GiB')
print('Stage shards locally:',STAGE_LOCAL)

cfg,phase2_root,data_root,registry,fr=bootstrap_campaign(PROJECT_ROOT,LOCAL_PACKAGE,stage_local=STAGE_LOCAL)
print('Phase-2 root:',phase2_root)
print('Data root   :',data_root)
print('Patch counts:',{k:len(v) for k,v in fr.items()})
print('Event counts:',{k:int(v.event_id.nunique()) for k,v in fr.items()})
print('Validation classes:',sorted(fr['val'].class_name.unique()))
print('B1-R ZERO RAW-SUPPORT OVERLAP GATE: PASS')


In [ ]:
# Model-contract, RDFT, and initialization-equivalence gates.
for seed in cfg.seeds:
    print(initialization_equivalence_gate(seed,cfg))

x=torch.randn(1,1,8192,128,device='cuda')
with torch.no_grad():
    m=build_model('raw_ts_resnet',9,192).cuda().eval(); logits,z=m(x,return_embedding=True)
assert logits.shape==(1,9) and z.shape==(1,192)
del m,logits,z

pre=RDFTPreprocessor().cuda().eval()
with torch.no_grad(): sx=pre(torch.zeros(1,1,8192,128,device='cuda'))
assert sx.shape==(1,1,2048,128) and torch.isfinite(sx).all()
fs=20000.0; t=torch.arange(8192,device='cuda')/fs; sig=torch.sin(2*torch.pi*100.0*t)
xx=sig[None,None,:,None].repeat(1,1,1,128)
with torch.no_grad(): ss=pre(xx)[0,0,:,0]
peak=int(torch.argmax(ss).item()); expected=round(100.0/(fs/(8192*3)))
print('100-Hz expected bin:',expected,'observed:',peak)
assert abs(peak-expected)<=1
del pre,sx,xx,ss,x
torch.cuda.empty_cache()
print('MODEL / RDFT / INITIALIZATION GATES: PASS')


In [ ]:
# Stage 1: Raw TS-ResNet + Raw SCAL-DAS, seed 2026 smoke/verification gate.
# Smoke uses train+validation only and does NOT construct/evaluate a test loader.
smoke_gate=run_smoke(fr,data_root,cfg,OUTPUT_ROOT)
print(json.dumps(smoke_gate,indent=2))
assert smoke_gate['status']=='PASS' and smoke_gate['test_accessed'] is False
print('B1-R RAW seed-2026 SMOKE/VERIFICATION GATE: PASS')


## Stage 2 — Full campaign

The next cell executes the frozen **4 models × 3 seeds × 60 epochs** campaign. It is resume-safe. If Colab disconnects, reconnect and rerun the notebook from the top; completed runs are skipped and incomplete runs resume from `resume_last.pt` stored in Google Drive. No test metric is used for training or checkpoint selection.

In [ ]:
runs=run_full(fr,data_root,cfg,OUTPUT_ROOT)
print('Completed campaign records:',len(runs))
per,agg=aggregate_campaign(OUTPUT_ROOT,cfg)
display(per.sort_values(['model','seed']))
display(agg)
out_zip,out_sha=make_audit_zip(OUTPUT_ROOT,LOCAL_PACKAGE)
print('\nB1-R RETRAINING CAMPAIGN: COMPLETE')
print('Audit ZIP:',out_zip)
print('Audit SHA256:',out_sha)
print('Upload this audit ZIP back to ChatGPT for independent audit and manuscript revision.')
